In [1]:
import xrfclk
import xrfdc
import pynq
from pynq import Overlay, MMIO
from pynq import lib
import numpy as np
import time
import os
import subprocess
import time 
import logging 

In [2]:
CLOCKWIZARD_LOCK_ADDRESS = 0x0004
CLOCKWIZARD_RESET_ADDRESS = 0x0000
CLOCKWIZARD_RESET_TOKEN = 0x000A
MTS_START_TILE = 0x01
MAX_DAC_TILES = 4
MAX_ADC_TILES = 4
DAC_REF_TILE = 2
ADC_REF_TILE = 2

RFSOC4X2_LMK_FREQ = 500.0
RFSOC4X2_LMX_FREQ = 500.0
RFSOC4X2_DAC_TILES = 0b0101
RFSOC4X2_ADC_TILES = 0b0101 

In [3]:
from pynq import MMIO
import time
ber = MMIO(0x80010000, 0x1000)
def estado():
    st   = ber.read(0x0C)
    bits = (ber.read(0x04) << 32) | ber.read(0x00)
    errs = ber.read(0x08)
    insync = (st >> 4) & 1
    rot    = (st >> 5) & 3
    val    = errs / bits if bits else float('nan')
    print("in_sync=%d  locked_rot=%d  bits=%d  errs=%d  BER=%.3g" % (insync, rot, bits, errs, val))
 

In [4]:
RFSOC4X2_LMK_FREQ = 500.0
RFSOC4X2_LMX_FREQ = 500.0
xrfclk.set_ref_clks(lmk_freq = RFSOC4X2_LMK_FREQ, lmx_freq = RFSOC4X2_LMX_FREQ)

In [5]:
ol = Overlay('/usr/local/share/pynq-venv/lib/python3.10/site-packages/pynq/overlays/Ruben/mts.bit',ignore_version=True)

In [6]:
ol.ACTIVE_DAC_TILES = RFSOC4X2_DAC_TILES
ol.ACTIVE_ADC_TILES = RFSOC4X2_ADC_TILES

ol.xrfdc = ol.usp_rf_data_converter_1
ol.xrfdc.mts_dac_config.RefTile = DAC_REF_TILE  # DAC tile distributing reference clock
ol.xrfdc.mts_adc_config.RefTile = ADC_REF_TILE  # ADC 

#INIT SYNC TILES
ol.xrfdc.mts_dac_config.Tiles = 0b0001 # turn only one tile on first
ol.xrfdc.mts_adc_config.Tiles = 0b0001
ol.xrfdc.mts_dac_config.SysRef_Enable = 1
ol.xrfdc.mts_adc_config.SysRef_Enable = 1
ol.xrfdc.mts_dac_config.Target_Latency = -1
ol.xrfdc.mts_adc_config.Target_Latency = -1

ol.xrfdc.mts_dac()
ol.xrfdc.mts_adc()

ol.clocktreeMTS.MTSclkwiz.mmio.write_reg(CLOCKWIZARD_RESET_ADDRESS, CLOCKWIZARD_RESET_TOKEN)

time.sleep(0.1)
# Reset only user selected DAC tiles
bitvector = ol.ACTIVE_DAC_TILES
for n in range(MAX_DAC_TILES):
    if (bitvector & 0x1):
        ol.xrfdc.dac_tiles[n].Reset()
    bitvector = bitvector >> 1
# Reset ADC FIFO of only user selected tiles - restarts MTS engine
for toggleValue in range(0,1):
    bitvector = ol.ACTIVE_ADC_TILES
    for n in range(MAX_ADC_TILES):
        if (bitvector & 0x1):
            ol.xrfdc.adc_tiles[n].SetupFIFOBoth(toggleValue)
        bitvector = bitvector >> 1
        

In [7]:
#SYNC TILES
dacTarget=-1
adcTarget=-1

if ol.ACTIVE_DAC_TILES > 0:
    ol.xrfdc.mts_dac_config.Tiles = ol.ACTIVE_DAC_TILES # group defined in binary 0b1111
    ol.xrfdc.mts_dac_config.SysRef_Enable = 1
    ol.xrfdc.mts_dac_config.Target_Latency = dacTarget 
    ol.xrfdc.mts_dac()
else:
    ol.xrfdc.mts_dac_config.Tiles = 0x0
    ol.xrfdc.mts_dac_config.SysRef_Enable = 0
    
if ol.ACTIVE_ADC_TILES > 0:
    ol.xrfdc.mts_adc_config.Tiles = ol.ACTIVE_ADC_TILES
    ol.xrfdc.mts_adc_config.SysRef_Enable = 1
    ol.xrfdc.mts_adc_config.Target_Latency = adcTarget
    ol.xrfdc.mts_adc()
else:
    ol.xrfdc.mts_adc_config.Tiles = 0x0
    ol.xrfdc.mts_adc_config.SysRef_Enable = 0

In [8]:
Mux_input=ol.axi_gpio_selec
Mux_input.write(0x0,0) #1 sec_fija, 0 PRBS

In [9]:
Mux_input.write(0x8, 0) # 0 signo error sin modificar 

In [10]:
#ol.ber_checker_axi_lite_0.write(0x0C,0)
#dir(ol.ber_checker_axi_lite_0)

In [11]:
ber.write(0x0C, 0x02)
ber.write(0x0C, 0x00)   # reinicia el contador (clr)

for _ in range(10):
    time.sleep(1)
    estado()

in_sync=0  locked_rot=0  bits=0  errs=0  BER=nan
in_sync=0  locked_rot=0  bits=0  errs=0  BER=nan
in_sync=0  locked_rot=0  bits=0  errs=0  BER=nan
in_sync=0  locked_rot=0  bits=0  errs=0  BER=nan
in_sync=0  locked_rot=0  bits=0  errs=0  BER=nan
in_sync=0  locked_rot=0  bits=0  errs=0  BER=nan
in_sync=0  locked_rot=0  bits=0  errs=0  BER=nan
in_sync=0  locked_rot=0  bits=0  errs=0  BER=nan
in_sync=0  locked_rot=0  bits=0  errs=0  BER=nan
in_sync=0  locked_rot=0  bits=0  errs=0  BER=nan


In [12]:
high_count = ol.axi_gpio_count.channel1.read()
low_count = ol.axi_gpio_count.channel2.read()
total_count = (high_count << 32) | low_count

high_error = ol.axi_gpio_error.channel1.read()
low_error = ol.axi_gpio_error.channel2.read()
total_errores = (high_error << 32) | low_error

BER=total_errores/total_count
print(f"Errores acumulados: {total_errores}")
print(f"Bits procesados: {total_count}")
print(f"BER actual: {BER:.2e}")

ZeroDivisionError: division by zero

In [ ]:
muestras=ol.axi_gpio_salidas.channel2.read()
muestras_4 = (muestras >> 6) & 0b11  # Los dos bits más significativos (izquierda)
muestras_3 = (muestras >> 4) & 0b11  # Siguientes dos bits
muestras_2 = (muestras >> 2) & 0b11  # Siguientes dos bits
muestras_1 = (muestras >> 0) & 0b11  # Los dos bits menos significativos (derecha)


print(f"Dato1: {muestras_1}")
print(f"Dato2: {muestras_2}")
print(f"Dato3: {muestras_3}")
print(f"Dato4: {muestras_4}")

In [ ]:
muestras=ol.axi_gpio_salidas.channel1.read()
print(f"Count: {muestras}")